# Capa Bronze - Ingesta

Lectura de los 4 CSV fuente desde Azure Blob Storage y persistencia en formato Delta sin transformaciones.

In [ ]:
BASE_VOL = "/Volumes/dbx_diplo_ricardo/default/clase8/clase10"

dbutils.fs.mkdirs(f"{BASE_VOL}/bronze")
dbutils.fs.ls(BASE_VOL)

## Configuración de acceso al Storage Account

In [ ]:
storage_account = "stdiploventas"
container       = "medallion"
account_key     = "TU_KEY_AQUI"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.blob.core.windows.net",
    account_key
)

BASE_BLOB = f"wasbs://{container}@{storage_account}.blob.core.windows.net"
dbutils.fs.ls(BASE_BLOB)

## Lectura de CSV y escritura Delta en Bronze

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("BronzeIngesta").getOrCreate()

ventas_bronze = spark.read.option("header", True).csv(f"{BASE_BLOB}/Ventas_diarias.csv")
ventas_bronze.write.format("delta").mode("overwrite").save(f"{BASE_VOL}/bronze/ventas_diarias")
print("ventas_diarias:", ventas_bronze.count())

clientes_bronze = spark.read.option("header", True).csv(f"{BASE_BLOB}/Clientes.csv")
clientes_bronze.write.format("delta").mode("overwrite").save(f"{BASE_VOL}/bronze/clientes")
print("clientes:", clientes_bronze.count())

productos_bronze = spark.read.option("header", True).csv(f"{BASE_BLOB}/Productos.csv")
productos_bronze.write.format("delta").mode("overwrite").save(f"{BASE_VOL}/bronze/productos")
print("productos:", productos_bronze.count())

tiendas_bronze = spark.read.option("header", True).csv(f"{BASE_BLOB}/tiendas.csv")
tiendas_bronze.write.format("delta").mode("overwrite").save(f"{BASE_VOL}/bronze/tiendas")
print("tiendas:", tiendas_bronze.count())

## Registro de tablas Bronze en Unity Catalog

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS dbx_diplo_ricardo.bronze_clase10")

for tabla in ["ventas_diarias", "clientes", "productos", "tiendas"]:
    spark.sql(f"DROP TABLE IF EXISTS dbx_diplo_ricardo.bronze_clase10.bronze_{tabla}")
    spark.sql(f"""
        CREATE TABLE dbx_diplo_ricardo.bronze_clase10.bronze_{tabla}
        USING DELTA
        LOCATION '{BASE_VOL}/bronze/{tabla}'
    """)

spark.sql("SHOW TABLES IN dbx_diplo_ricardo.bronze_clase10").show(truncate=False)

## Verificación

In [ ]:
for tabla in ["ventas_diarias", "clientes", "productos", "tiendas"]:
    print("=" * 60)
    print("BRONZE -", tabla)
    df = spark.read.format("delta").load(f"{BASE_VOL}/bronze/{tabla}")
    df.printSchema()
    df.show(5, truncate=False)

In [ ]:
%sql
SELECT * FROM dbx_diplo_ricardo.bronze_clase10.bronze_ventas_diarias LIMIT 10;